In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
import torch.nn.functional as F
from torchvision import datasets, transforms, models


import matplotlib.pyplot as plt

from sklearn.utils import shuffle
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

rng = np.random.RandomState(1234)
random_state = 42


In [2]:
batch_size = 1  

dataloader_train = torch.utils.data.DataLoader(
    datasets.CIFAR10('./data/cifar10', train=True, download=True, transform=transforms.ToTensor()),
    batch_size=batch_size,
    shuffle=False
)

 77%|███████▋  | 131M/170M [2:25:39<44:32, 14.9kB/s]   


RuntimeError: File not found or corrupted.

In [3]:
# 学習する前の前処理を行う
class gcn():
    def __init__(self):
        pass

    def __call__(self, x):
        mean = torch.mean(x)
        std = torch.std(x)
        return (x - mean)/(std + 10**(-6)) 



def deprocess(x):
    """
    Argument
    --------
    x : np.ndarray
        input image．(H, W, C)

    Return
    ------
    _x : np.ndarray
        Image normalized to [0, 1]．(H, W, C)
    """
    _min = np.min(x)
    _max = np.max(x)
    _x = (x - _min)/(_max - _min)
    return _x

In [4]:
# ZCAの実装
class ZCAWhitening():
    def __init__(self, epsilon=1e-4, device="cuda"): 
        self.epsilon = epsilon
        self.device = device

    def fit(self, images):  
        """
        Argument
        --------
        images : torchvision.datasets.cifar.CIFAR10
            input image（Entire training data）．(N, C, H, W)
        """
        x = images[0][0].reshape(1, -1)  
        self.mean = torch.zeros([1, x.size()[1]]).to(self.device)
        con_matrix = torch.zeros([x.size()[1], x.size()[1]]).to(self.device)
        for i in range(len(images)):  
            x = images[i][0].reshape(1, -1).to(self.device)
            self.mean += x / len(images)
            con_matrix += torch.mm(x.t(), x) / len(images)
            if i % 10000 == 0:
                print("{0}/{1}".format(i, len(images)))
        con_matrix -= torch.mm(self.mean.t(), self.mean)
        
        E, V = torch.linalg.eigh(con_matrix)  
        self.ZCA_matrix = torch.mm(torch.mm(V, torch.diag((E.squeeze()+self.epsilon)**(-0.5))), V.t())  
        print("completed!")

    def __call__(self, x):
        size = x.size()
        x = x.reshape(1, -1).to(self.device)
        x -= self.mean  # x - \bar{x}
        x = torch.mm(x, self.ZCA_matrix.t())
        x = x.reshape(tuple(size))
        x = x.to("cpu")
        return x

In [5]:
# バッチ正規化の実装
class BatchNorm(nn.Module):
    def __init__(self, shape, epsilon=np.float32(1e-5)):
        super().__init__()
        self.gamma = nn.Parameter(torch.tensor(np.ones(shape, dtype='float32')))
        self.beta = nn.Parameter(torch.tensor(np.zeros(shape, dtype='float32')))
        self.epsilon = epsilon

    def forward(self, x):
        mean = torch.mean(x, (0, 2, 3), keepdim=True)   # WRITE ME # You can calculate the average with torch.mean. In addition, for image data, calculations are made for each channel.
        std = torch.std(x, (0, 2, 3), keepdim=True)   # WRITE ME # Standard deviation can be calculated with torch.std
        x_normalized = (x - mean) / (std**2 + self.epsilon)**0.5 # WRITE ME
        return self.gamma * x_normalized + self.beta # WRITE ME

In [6]:
# Dropoutの実装
class Dropout(nn.Module):
    """
    http://arxiv.org/abs/1207.0580
    """
    def __init__(self, dropout_ratio=0.5):
        super().__init__()
        self.dropout_ratio = dropout_ratio
        self.mask = None

    def forward(self, x):
        # Shuts down output by dropout_ratio during learning
        if self.training:
            self.mask = torch.rand(*x.size()) > self.dropout_ratio
            return x * self.mask.to(x.device)
        # During inference, multiply the output by `1.0 - self.dropout_ratio` to match the size of the output during training.
        else:
            return x * (1.0 - self.dropout_ratio)

In [7]:
# nn.Moduleを継承して、PyTorchの層として使えるようにする
class Conv(nn.Module):
    # filter_shape：フィルターの形状
    # function：活性化関数
    # stride：ストライド
    # padding：パディング
    def __init__(self, filter_shape, function=lambda x: x, stride=(1, 1), padding=0):
        # 親クラス（nn.Module）の初期化
        super().__init__()
        
        # fan_in；1つのフィルターが一度に受け取る情報の数
        # filter_shaoe[1]：入力チャンネル数
        # filter_shape[2]：フィルターの高さ
        # fikter_shape[3]：フィルターの幅
        fan_in = filter_shape[1] * filter_shape[2] * filter_shape[3]
        # filter_out：次の層へ渡す出力のサイズ
        # filter_shape[0]：フィルターの枚数
        fan_out = filter_shape[0] * filter_shape[2] * filter_shape[3]

        # 重みの初期値
        self.W = nn.Parameter(torch.tensor(rng.normal(
                        0,
                        np.sqrt(2/fan_in),
                        size=filter_shape
                    ).astype('float32')))

        # バイアスを0で初期化する
        self.b = nn.Parameter(torch.tensor(np.zeros((filter_shape[0]), dtype='float32')))

        # クラス内でいつでも呼び出せるよう、インスタンス変数として保持
        self.function = function  # activation function
        self.stride = stride  # stride width
        self.padding = padding  # padding

    def forward(self, x):
        # 畳み込み計算
        u = F.conv2d(x, self.W, bias=self.b, stride=self.stride, padding=self.padding)
        # 活性化関数を通して、最終的な出力を計算
        return self.function(u)

In [8]:
class Pooling(nn.Module):
    def __init__(self, ksize=(2, 2), stride=(2, 2), padding=0):
        super().__init__()
        # プーリングを行う範囲
        self.ksize = ksize  # kernel size
        # ストライド
        self.stride = stride  # stride width
        # パディング
        self.padding = padding  # padding

    def forward(self, x):
        return F.avg_pool2d(x, kernel_size=self.ksize, stride=self.stride, padding=self.padding)

In [9]:
# 全結合層の実装
class Dense(nn.Module):
    def __init__(self, in_dim, out_dim, function=lambda x: x):
        super().__init__()
    
        self.W = nn.Parameter(torch.tensor(rng.normal(
                        0,
                        np.sqrt(2/in_dim),
                        size=(in_dim, out_dim)
                    ).astype('float32')))

        self.b = nn.Parameter(torch.tensor(np.zeros([out_dim]).astype('float32')))
        self.function = function

    def forward(self, x):
        return self.function(torch.matmul(x, self.W) + self.b)

In [10]:
class Activation(nn.Module):
    def __init__(self, function=lambda x: x):
        super().__init__()
        self.function = function

    def __call__(self, x):
        return self.function(x)